# Hybrid Search QA Assistant

**Hardware:** Google Colab free GPU (T4)

### System Overview
- **BM25** — keyword lexical search
- **all-MiniLM-L6-v2** — dense semantic search
- **RRF** — Reciprocal Rank Fusion to combine both
- **OpenRouter (openai/gpt-oss-120b:free)** — free LLM for answer generation with citations


## Setup: Install Dependencies


In [ ]:
# Install all required packages
!pip install rank_bm25 sentence-transformers openai torch numpy


## Data Loading and Preprocessing

**Upload both JSON files** to Colab using the 📁 Files panel:
- `top_ai_questions.json`
- `top_datascience_questions.json`


In [ ]:
import json, re, numpy as np

def clean_html(text):
    '''Strips HTML tags from raw StackExchange post body.'''
    return re.sub(re.compile('<.*?>'), '', text).strip()

# Paths to uploaded dataset files
data_paths = ["top_ai_questions.json", "top_datascience_questions.json"]

documents = []
for path in data_paths:
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for item in json.load(f):
                title = item.get('title', '')
                body  = clean_html(item.get('body', ''))
                # Combine title and body for a single searchable text field
                documents.append({
                    'id':    item.get('question_id', ''),
                    'title': title,
                    'text':  f"{title}. {body}",
                    'link':  item.get('link', '')
                })
    except FileNotFoundError:
        print(f"Not found: {path} — please upload it.")

doc_texts = [d['text'] for d in documents]
print(f"Total documents loaded: {len(documents)}")



## Milestone 1: BM25 Retrieval

BM25 (Best Match 25) is a probabilistic term-frequency ranking function. It scores documents by how often query terms appear, weighted by inverse document frequency.


In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize corpus for BM25 (lowercase + whitespace split)
tokenized_corpus = [d.lower().split() for d in doc_texts]
bm25 = BM25Okapi(tokenized_corpus)

def search_bm25(query, top_k=10):
    '''Retrieves top_k documents by BM25 lexical score.'''
    scores  = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [{'doc_id': documents[i]['id'], 'title': documents[i]['title'],
             'score': scores[i], 'rank': r + 1}
            for r, i in enumerate(top_idx)]

# --- Sample query ---
sample_q = "What is the difference between machine learning and AI?"
print(f"Query: {sample_q}\n")
for r in search_bm25(sample_q, top_k=10):
    print(f"  Rank {r['rank']:2d}: {r['title']}  (BM25={r['score']:.2f})")



## Milestone 2: Retrieval Evaluation — MAP@10 and MRR@10

- **MRR@10** rewards finding the first relevant result as early as possible.
- **MAP@10** measures average precision across all relevant results.

> Ground truth uses known StackExchange question IDs matching the queries.


In [ ]:
# Ground truth: query → list of known relevant question IDs
ground_truth = {
    "What is the difference between machine learning and AI?": [35],
    "How do capsule neural networks work?": [1294]
}

def mrr(results, rel_ids):
    '''Mean Reciprocal Rank for a single query.'''
    for r in results:
        if r['doc_id'] in rel_ids:
            return 1.0 / r['rank']
    return 0.0

def ap(results, rel_ids):
    '''Average Precision for a single query.'''
    hits, prec_sum = 0, 0
    for r in results:
        if r['doc_id'] in rel_ids:
            hits += 1
            prec_sum += hits / r['rank']
    return prec_sum / len(rel_ids) if rel_ids else 0.0

# Compute MAP@10 and MRR@10
mrr_sum = map_sum = 0
for q, rel in ground_truth.items():
    res = search_bm25(q, top_k=10)
    mrr_sum += mrr(res, rel)
    map_sum += ap(res, rel)

n = len(ground_truth)
print(f"BM25  MRR@10 : {mrr_sum / n:.4f}")
print(f"BM25  MAP@10 : {map_sum / n:.4f}")



## Milestone 3: Semantic Search + Hybrid Fusion (RRF)

Dense embeddings from **all-MiniLM-L6-v2** are combined with BM25 rankings using **Reciprocal Rank Fusion**: $\text{RRF}(d)=\sum_r \frac{1}{60+r(d)}$

We also compute **nDCG@10** to compare BM25 vs Semantic retrieval quality.


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch, math

# Load the lightweight MiniLM model (90 MB)
st_model = SentenceTransformer('all-MiniLM-L6-v2')

# Pre-compute dense embeddings for all documents (~1-2 min on T4 GPU)
print("Computing semantic embeddings...")
doc_embeddings = st_model.encode(doc_texts, convert_to_tensor=True, show_progress_bar=True)

def search_semantic(query, top_k=10):
    '''Dense retrieval using cosine similarity of sentence embeddings.'''
    q_emb  = st_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, doc_embeddings)[0]
    top    = torch.topk(scores, k=top_k)
    return [{'doc_id': documents[i]['id'], 'title': documents[i]['title'],
             'score': s.item(), 'rank': r + 1}
            for r, (s, i) in enumerate(zip(top[0], top[1]))]

def rrf(rank, k=60):
    '''Reciprocal Rank Fusion score.'''
    return 1 / (k + rank)

def search_hybrid(query, top_k=10):
    '''Fuses BM25 + Semantic via RRF.'''
    combined, docs_map = {}, {}
    for res in search_bm25(query, 50) + search_semantic(query, 50):
        did = res['doc_id']
        docs_map[did] = res
        combined[did] = combined.get(did, 0) + rrf(res['rank'])
    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{'doc_id': did, 'title': docs_map[did]['title'], 'score': s, 'rank': r+1}
            for r, (did, s) in enumerate(ranked)]

# --- Test hybrid search ---
print("\nHybrid Search Results:\n")
for r in search_hybrid(sample_q, top_k=5):
    print(f"  Rank {r['rank']}: {r['title']}  (RRF={r['score']:.4f})")

# --- nDCG@10 comparison ---
def ndcg(results, rel_ids, k=10):
    '''Normalised Discounted Cumulative Gain at k.'''
    dcg  = sum(1/math.log2(r['rank']+1) for r in results[:k] if r['doc_id'] in rel_ids)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(rel_ids), k)))
    return dcg / idcg if idcg else 0

nb, ns = 0, 0
for q, rel in ground_truth.items():
    nb += ndcg(search_bm25(q, 10), rel)
    ns += ndcg(search_semantic(q, 10), rel)
n = len(ground_truth)
print(f"\nBM25     nDCG@10: {nb/n:.4f}")
print(f"Semantic nDCG@10: {ns/n:.4f}")



## Milestone 4 & 5: RAG with OpenRouter

We use **OpenRouter** to access the `openai/gpt-oss-120b:free` model.

**Get your free API key:**
1. Go to [https://openrouter.ai](https://openrouter.ai)
2. Create an API key and paste it below.


In [ ]:
from openai import OpenAI

# 🔑 PASTE YOUR FREE OPENROUTER KEY HERE
OPENROUTER_API_KEY = "sk-or-v1-YOUR_KEY_HERE"
OPENROUTER_MODEL = "openai/gpt-oss-120b:free"

# Initialize OpenRouter client
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)
print(f"OpenRouter client ready. Model: {OPENROUTER_MODEL}")



In [ ]:
def generate_answer(query, top_k=3):
    '''Generates an answer using retrieved context and OpenRouter.'''
    # 1. Retrieve relevant documents
    retrieved = search_hybrid(query, top_k=top_k)

    # 2. Format context with sources
    source_blocks, sources = [], []
    for i, res in enumerate(retrieved):
        doc = next(d for d in documents if d['id'] == res['doc_id'])
        snippet = doc['text'][:500].replace('\n', ' ')
        source_blocks.append(f"[Source {i+1}] Title: {doc['title']}\nContent: {snippet}")
        sources.append(doc['title'])

    # 3. Create prompt
    context = "\n\n".join(source_blocks)
    messages = [
        {"role": "system", "content": "You are a technical assistant. Answer using ONLY provided sources. Cite like [Source 1]."},
        {"role": "user", "content": f"Sources:\n{context}\n\nQuestion: {query}\n\nAnswer:"}
    ]

    # 4. Generate response
    response = client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        max_tokens=400,
        temperature=0.3,
    )

    print(f"[OpenRouter] Model actually used: {response.model}")
    return response.choices[0].message.content.strip(), sources

# ── Evaluate test questions ──────────────────────────────────────────
questions = [
    "What is the difference between strong-AI and weak-AI?",
    "How do capsule neural networks work?",
    "Why does the transformer do better than RNN?",
    "What is fuzzy logic?",
    "Was ChatGPT trained on Stack Overflow data?"
]

for q in questions:
    ans, srcs = generate_answer(q)
    print(f"\nQ: {q}\nA: {ans}\nSources: {srcs}\n" + "─"*60)

